# Reusable Customer Churn Prediction Pipeline

This notebook builds a reusable scikit-learn pipeline for customer churn prediction. It includes data loading, validation, train-test split, preprocessing, model training, evaluation, saving the pipeline, and making a prediction for a new customer.

## Import Libraries

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

## Load and Inspect Dataset

In [ ]:
df = pd.read_csv("customer_churn.csv")

display(df.head())
print(df.info())
print("Missing values:")
print(df.isnull().sum())
print("Churn distribution:")
print(df["Churn"].value_counts())

## Prepare Features and Target

`CustomerID` is removed because it is only an identifier. `X` contains the input features and `y` contains the target column.

In [ ]:
df_model = df.drop("CustomerID", axis=1)

X = df_model.drop("Churn", axis=1)
y = df_model["Churn"]

## Train-Test Split

`test_size=0.2` creates an 80:20 split: 80% for training and 20% for testing. `stratify=y` keeps the churn ratio similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## Build Reusable Pipeline

In [ ]:
numeric_features = [
    "Age",
    "Tenure",
    "MonthlyCharges",
    "TotalCharges",
    "SupportTickets",
]

categorical_features = [
    "Gender",
    "ContractType",
    "PaymentMethod",
    "InternetService",
]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(random_state=42)),
    ]
)

## Train and Evaluate

In [ ]:
model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))

## Save and Reuse Pipeline

In [ ]:
joblib.dump(model_pipeline, "customer_churn_pipeline.pkl")

loaded_pipeline = joblib.load("customer_churn_pipeline.pkl")

new_customer = pd.DataFrame({
    "Gender": ["Female"],
    "Age": [32],
    "Tenure": [5],
    "MonthlyCharges": [850],
    "TotalCharges": [4250],
    "ContractType": ["Monthly"],
    "PaymentMethod": ["Credit Card"],
    "InternetService": ["Fiber"],
    "SupportTickets": [3],
})

prediction = loaded_pipeline.predict(new_customer)
print("New Customer Churn Prediction:", prediction[0])

## Business Interpretation

The model helps the subscription business identify customers who are likely to leave. Those customers can be prioritized for retention actions such as discounts, support follow-up, renewal offers, or personalized plans. For churn prediction, recall for the `Yes` class is especially important because missing a likely churn customer can lead to lost revenue.